In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [2]:
!git clone https://github.com/yoosufaamir/HPC-.git
!ls -la HPC-

Cloning into 'HPC-'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 698.61 KiB | 5.10 MiB/s, done.
Resolving deltas: 100% (7/7), done.
total 1176
drwxr-xr-x 4 root root    4096 Oct  8 18:14 .
drwxr-xr-x 1 root root    4096 Oct  8 18:14 ..
drwxr-xr-x 8 root root    4096 Oct  8 18:14 .git
drwxr-xr-x 2 root root    4096 Oct  8 18:14 images
-rw-r--r-- 1 root root   24181 Oct  8 18:14 MatData.txt
-rw-r--r-- 1 root root  110000 Oct  8 18:14 passwords.txt
-rw-r--r-- 1 root root       6 Oct  8 18:14 README.md
-rw-r--r-- 1 root root 1048138 Oct  8 18:14 WordOccurrenceDataset.txt


In [6]:
%%writefile matrix_ops.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <omp.h>

#define MAX_FILE_NAME 200
#define OUTPUT_FILE "results.txt"

typedef struct {
    int rows;
    int cols;
    double **data;   /* allocated dynamically with malloc */
} Matrix;



Matrix *readMatricesFromFile(const char *filename, int *outCount) {

    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        fprintf(stderr, "Error: could not open input file '%s'\n", filename);
        *outCount = -1;
        return NULL;
    }

    int capacity = 8;
    int count = 0;
    Matrix *matrices = malloc(capacity * sizeof(Matrix));

    int rows, cols;


    while (fscanf(fp, "%d,%d", &rows, &cols) == 2) {

        if (rows <= 0 || cols <= 0) {
            fprintf(stderr, "Error: invalid matrix header (%d,%d) in file.\n", rows, cols);
            break;
        }

        double **data = malloc(rows * sizeof(double *));
        int validMatrix = 1;

        for (int r = 0; r < rows && validMatrix; r++) {
            data[r] = malloc(cols * sizeof(double));
            for (int c = 0; c < cols; c++) {
                double value;
                /* leading comma is optional - fscanf with " %lf,"
                 * skips whitespace/commas automatically */
                if (fscanf(fp, " %lf,", &value) != 1) {
                    fprintf(stderr,
                        "Error: bad/non-numeric value while reading matrix "
                        "%d (expected %d x %d values).\n", count + 1, rows, cols);
                    validMatrix = 0;
                    break;
                }
                data[r][c] = value;
            }
        }

        if (!validMatrix) {
            /* clean up the partially read matrix before exiting */
            for (int r = 0; r < rows; r++) free(data[r]);
            free(data);
            break;
        }

        if (count >= capacity) {
            capacity *= 2;
            matrices = realloc(matrices, capacity * sizeof(Matrix));
        }

        matrices[count].rows = rows;
        matrices[count].cols = cols;
        matrices[count].data = data;
        count++;
    }

    fclose(fp);
    *outCount = count;
    return matrices;
}

void freeMatrix(Matrix *m) {
    for (int r = 0; r < m->rows; r++) {
        free(m->data[r]);
    }
    free(m->data);
}



int cappedThreads(int requested, int maxDim) {
    return (requested > maxDim) ? maxDim : requested;
}

Matrix allocResult(int rows, int cols) {
    Matrix m;
    m.rows = rows;
    m.cols = cols;
    m.data = malloc(rows * sizeof(double *));
    for (int r = 0; r < rows; r++) {
        m.data[r] = malloc(cols * sizeof(double));
    }
    return m;
}

void writeMatrixToFile(FILE *out, const char *opName, Matrix *m) {
    fprintf(out, "%s - %d,%d\n", opName, m->rows, m->cols);
    for (int r = 0; r < m->rows; r++) {
        for (int c = 0; c < m->cols; c++) {
            fprintf(out, "%.4f", m->data[r][c]);
            if (c < m->cols - 1) fprintf(out, ",");
        }
        fprintf(out, "\n");
    }
    fprintf(out, "\n");
}



Matrix matAdd(Matrix *A, Matrix *B, int threadCount) {
    Matrix R = allocResult(A->rows, A->cols);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j;
    #pragma omp parallel for shared(A, B, R) private(i, j)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < A->cols; j++) {
            R.data[i][j] = A->data[i][j] + B->data[i][j];
        }
    }
    return R;
}

Matrix matSub(Matrix *A, Matrix *B, int threadCount) {
    Matrix R = allocResult(A->rows, A->cols);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j;
    #pragma omp parallel for shared(A, B, R) private(i, j)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < A->cols; j++) {
            R.data[i][j] = A->data[i][j] - B->data[i][j];
        }
    }
    return R;
}

Matrix matElementMultiply(Matrix *A, Matrix *B, int threadCount) {
    Matrix R = allocResult(A->rows, A->cols);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j;
    #pragma omp parallel for shared(A, B, R) private(i, j)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < A->cols; j++) {
            R.data[i][j] = A->data[i][j] * B->data[i][j];
        }
    }
    return R;
}

Matrix matElementDivide(Matrix *A, Matrix *B, int threadCount) {
    Matrix R = allocResult(A->rows, A->cols);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j;
    #pragma omp parallel for shared(A, B, R) private(i, j)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < A->cols; j++) {
            if (B->data[i][j] == 0.0) {
                R.data[i][j] = NAN;     /* divisor is zero -> write NaN, keep going */
            } else {
                R.data[i][j] = A->data[i][j] / B->data[i][j];
            }
        }
    }
    return R;
}

Matrix matTranspose(Matrix *A, int threadCount) {
    Matrix R = allocResult(A->cols, A->rows);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j;
    #pragma omp parallel for shared(A, R) private(i, j)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < A->cols; j++) {
            R.data[j][i] = A->data[i][j];
        }
    }
    return R;
}

Matrix matMultiply(Matrix *A, Matrix *B, int threadCount) {
    Matrix R = allocResult(A->rows, B->cols);
    int threads = cappedThreads(threadCount, A->rows);
    omp_set_num_threads(threads);

    int i, j, k;
    #pragma omp parallel for shared(A, B, R) private(i, j, k)
    for (i = 0; i < A->rows; i++) {
        for (j = 0; j < B->cols; j++) {
            double sum = 0.0;
            for (k = 0; k < A->cols; k++) {
                sum += A->data[i][k] * B->data[k][j];
            }
            R.data[i][j] = sum;
        }
    }
    return R;
}


int main(int argc, char *argv[]) {

    if (argc != 3) {
        printf("Usage: %s <matrixFile> <threadCount>\n", argv[0]);
        return 1;
    }

    char filename[MAX_FILE_NAME];
    strncpy(filename, argv[1], MAX_FILE_NAME - 1);
    int requestedThreads = atoi(argv[2]);

    if (requestedThreads <= 0) {
        printf("threadCount must be a positive integer.\n");
        return 1;
    }

    int numMatrices;
    Matrix *matrices = readMatricesFromFile(filename, &numMatrices);

    if (numMatrices <= 0) {
        printf("No valid matrices were read from '%s'. Exiting.\n", filename);
        return 1;
    }

    printf("Read %d matrices from %s\n", numMatrices, filename);

    FILE *out = fopen(OUTPUT_FILE, "w");
    if (out == NULL) {
        perror("Error opening output file");
        for (int i = 0; i < numMatrices; i++) freeMatrix(&matrices[i]);
        free(matrices);
        return 1;
    }

    int pairNum = 0;

    for (int i = 0; i + 1 < numMatrices; i += 2) {

        Matrix *A = &matrices[i];
        Matrix *B = &matrices[i + 1];
        pairNum++;

        fprintf(out, "================ Pair %d (Matrix %d & Matrix %d) ================\n",
                pairNum, i + 1, i + 2);
        fprintf(out, "Matrix A: %d,%d   Matrix B: %d,%d\n\n",
                A->rows, A->cols, B->rows, B->cols);

        int sameShape = (A->rows == B->rows) && (A->cols == B->cols);
        int multOK = (A->cols == B->rows);

        /* Addition */
        if (sameShape) {
            Matrix R = matAdd(A, B, requestedThreads);
            writeMatrixToFile(out, "Addition", &R);
            freeMatrix(&R);
        } else {
            fprintf(out, "Addition cannot be done (shapes differ).\n\n");
        }

        /* Subtraction */
        if (sameShape) {
            Matrix R = matSub(A, B, requestedThreads);
            writeMatrixToFile(out, "Subtraction", &R);
            freeMatrix(&R);
        } else {
            fprintf(out, "Subtraction cannot be done (shapes differ).\n\n");
        }

        /* Element-wise multiply */
        if (sameShape) {
            Matrix R = matElementMultiply(A, B, requestedThreads);
            writeMatrixToFile(out, "Element-wise Multiplication", &R);
            freeMatrix(&R);
        } else {
            fprintf(out, "Element-wise multiplication cannot be done (shapes differ).\n\n");
        }

        /* Element-wise divide */
        if (sameShape) {
            Matrix R = matElementDivide(A, B, requestedThreads);
            writeMatrixToFile(out, "Element-wise Division", &R);
            freeMatrix(&R);
        } else {
            fprintf(out, "Element-wise division cannot be done (shapes differ).\n\n");
        }

        /* Transpose A and Transpose B - always valid */
        {
            Matrix RA = matTranspose(A, requestedThreads);
            writeMatrixToFile(out, "Transpose(A)", &RA);
            freeMatrix(&RA);

            Matrix RB = matTranspose(B, requestedThreads);
            writeMatrixToFile(out, "Transpose(B)", &RB);
            freeMatrix(&RB);
        }

        /* Matrix multiplication A x B */
        if (multOK) {
            Matrix R = matMultiply(A, B, requestedThreads);
            writeMatrixToFile(out, "Matrix Multiplication (A x B)", &R);
            freeMatrix(&R);
        } else {
            fprintf(out, "Matrix multiplication cannot be done (A.cols != B.rows).\n\n");
        }

        printf("Processed pair %d (%d,%d) & (%d,%d)\n",
               pairNum, A->rows, A->cols, B->rows, B->cols);
    }


    if (numMatrices % 2 != 0) {
        fprintf(out, "Note: matrix %d had no partner and was not processed.\n", numMatrices);
        printf("Note: matrix %d had no partner and was skipped.\n", numMatrices);
    }

    fclose(out);
    printf("All pairs processed. Results written to %s\n", OUTPUT_FILE);


    for (int i = 0; i < numMatrices; i++) {
        freeMatrix(&matrices[i]);
    }
    free(matrices);

    return 0;
}

Overwriting matrix_ops.c


In [7]:
!gcc matrix_ops.c -o matrix_ops -fopenmp -lm

In [8]:
!gcc matrix_ops.c -o matrix_ops -fopenmp -lm
!./matrix_ops HPC-/MatData.txt 4

Read 50 matrices from HPC-/MatData.txt
Processed pair 1 (3,4) & (3,4)
Processed pair 2 (4,6) & (6,2)
Processed pair 3 (5,5) & (5,5)
Processed pair 4 (8,4) & (4,8)
Processed pair 5 (6,6) & (6,6)
Processed pair 6 (7,3) & (7,3)
Processed pair 7 (10,8) & (8,10)
Processed pair 8 (6,4) & (6,4)
Processed pair 9 (12,12) & (12,12)
Processed pair 10 (5,7) & (7,2)
Processed pair 11 (4,4) & (4,4)
Processed pair 12 (6,6) & (3,3)
Processed pair 13 (4,5) & (5,4)
Processed pair 14 (7,3) & (7,3)
Processed pair 15 (6,8) & (8,6)
Processed pair 16 (6,4) & (6,4)
Processed pair 17 (5,5) & (5,5)
Processed pair 18 (3,3) & (3,3)
Processed pair 19 (9,5) & (5,9)
Processed pair 20 (3,6) & (3,6)
Processed pair 21 (4,4) & (4,4)
Processed pair 22 (6,2) & (2,6)
Processed pair 23 (8,4) & (4,8)
Processed pair 24 (6,6) & (6,6)
Processed pair 25 (10,10) & (10,10)
All pairs processed. Results written to results.txt
